# Imports

In [ ]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T

# Connect to Spark

In [ ]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [ ]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [ ]:
api.start_spark(n_executors=400, config=config)

In [ ]:
year0 = "2019"

In [ ]:
year1 = str(int(year0) + 1)
#year1 = year0

# Set Parameters

Set date range and airports here.

In [ ]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 + "-01-01"}

In [ ]:
airports = [
    "KADW",
    "KATL",
    "KBOS",
    "KBWI",
    "KCLT",
    "KDCA",
    "KDEN",
    "KDFW",
    "KDTW",
    "KEWR",
    "KFLL",
    "KIAD",
    "KIAH",
    "KJFK",
    "KLAS",
    "KLAX",
    "KLGA",
    "KMCO",
    "KMDW",
    "KMEM",
    "KMIA",
    "KMSP",
    "KORD",
    "KPHL",
    "KPHX",
    "KSAN",
    "KSDF",
    "KSEA",
    "KSFO",
    "KSLC",
    "KTPA",
    "PANC",
    "PHNL",
]

In [ ]:
#airports = [ "KADW", ]

# UDFs

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [ ]:
@F.udf("string")
def to_date(ts):
    return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

Define the schema of the litetrack points structure array

In [ ]:
points_schema = T.StructType([
    T.StructField("points", T.ArrayType(
        T.StructType([
            T.StructField("primary_key", T.StringType(), True),
            T.StructField("time", T.LongType(), True),
            T.StructField("latitude", T.DoubleType(), True),
            T.StructField("longitude", T.DoubleType(), True),
            T.StructField("altitude", T.FloatType(), True),
            T.StructField("course", T.FloatType(), True),
            T.StructField("speed", T.FloatType(), True),
            T.StructField("source_point_keys", T.ArrayType(
                T.StructType([
                    T.StructField("element", T.StringType(), True),
                ]), True), True),
            T.StructField("acceleration", T.DoubleType(),True),
            T.StructField("type", T.StringType(), True),
            T.StructField("airspace_key", T.StringType(), True),
            T.StructField("tas", T.FloatType(), True),
            T.StructField("ias", T.FloatType(), True),
            T.StructField("along_track_distance", T.FloatType(), True),
            T.StructField("derived_point_key", T.StringType(), True),
        ]), True), True)
    ])

Define a function that will convert the points array into a string containing n-tuples of data for each litetrack point

In [ ]:
none_string = "*"
separator_string = ":"

def stringify_points(array_of_points:T.ArrayType(points_schema)) -> str:
    output = ""
    for x in array_of_points:
        output += (none_string if x.time is None else str(x.time)) + " "
        output += (none_string if x.latitude is None else f'{x.latitude:.6f}') + " "
        output += (none_string if x.longitude is None else f'{x.longitude:.6f}') + " "
        output += (none_string if x.altitude is None else f'{x.altitude:.0f}') + " "
        output += (none_string if x.course is None else f'{x.course:.0f}') + " "
        output += (none_string if x.speed is None else f'{x.speed:.0f}') + separator_string

    # drop the trailing tuple separator
    if len(array_of_points) > 0:
        output = output[0:len(separator_string) * -1]
    return output

stringify_points_udf = F.udf(stringify_points, T.StringType())

# Load LiteTrack

the points array is "stringified" to a new `trackpoints` column

In [ ]:
#df_lt_meta = api.dataframe(
#    "LiteTrack", **dates, partition_filters=api.custom_partitions("ASSOCIATED")
#).select(
#    "track_key",
#    F.col("start_time").alias("start_epoch_millisec"),
#    F.col("end_time").alias("end_epoch_millisec"),
#)

In [ ]:
df_lt = (
    api.dataframe(
        "LiteTrack", **dates, partition_filters=api.custom_partitions("ASSOCIATED")
    )
#    .withColumn("trackpoints", stringify_points_udf("points"))
    .select(
        "track_key",
        F.col("start_time").alias("start_epoch_millisec"),
        F.col("end_time").alias("end_epoch_millisec"),
#        "mode_s_code",
#        "trackpoints",
        "points",
    )
)

# Load Flightplan Series

Here, we'll load the FlightplanSeries and perform the following:

1. Filter flight plans so either the arrival or departure airport is for an airport we want
2. Ensure arrival and departure airports are not the same airport
3. Convert dates from unix time to YYYYMMDD using a previously defined UDF
4. Extract year and month from date (for partitioning later)
5. Join metadata from LiteTrack (only needed to add in start_time and end_time)

In [ ]:
df_fps = (
    api.dataframe("FlightplanSeries", **dates, metadata=True)
    .select(
        "track_key",
        to_date("metadata.effective_start_date").alias("start_date"),
        to_date("metadata.effective_end_date").alias("end_date"),
        "callsign",
        "aircraft_type",
        "mode_s_code",
        F.col("initial_departure_aerodrome").alias("orig"),
        F.col("final_destination_aerodrome").alias("dest"),
        "ffpd_route",
        "lfpd_route",
    )
    .withColumn("month", F.substring("start_date", 5, 2))
    .filter(F.col("orig") != F.col("dest"))
    .filter(F.col("orig").isin(airports) | F.col("dest").isin(airports))
    .join(df_lt, on=["track_key"], how="inner")
)

# Split out Arrivals and Departures

So we can save data into files pertaining to arrivals and departures, we'll:

1. Split data into arrivals and departures sets, adding an `operation column` to each and marking it as `ARRIVALS` or `DEPARTURES`
2. Re-apply filter to only airports we care about
3. Merge them back into one dataset afterwards

In [ ]:
df_arrivals = (
    df_fps.withColumn("airport", F.col("dest"))
    .withColumn("operation", F.lit("ARRIVALS"))
    .filter(F.col("airport").isin(airports))
)

In [ ]:
df_departures = (
    df_fps.withColumn("airport", F.col("orig"))
    .withColumn("operation", F.lit("DEPARTURES"))
    .filter(F.col("airport").isin(airports))
)

In [ ]:
df_fps_combined = df_arrivals.unionByName(df_departures)

# Save FlightplanSeries Data

create the fps output dataframe

In [ ]:
df_fps_output = (
    df_fps_combined
    .select(
        "track_key",
        "start_date",
        "end_date",
        "start_epoch_millisec",
        "end_epoch_millisec",
        "orig",
        "dest",
        "callsign",
        "mode_s_code",
        "aircraft_type",
        "ffpd_route",
        "lfpd_route",
        "airport",
        "month",
        "operation",
    )
)

Here, we'll partition by a few fields:

`AIRPORT / MONTH / OPERATION TYPE`

the .repartition(cols) "...shuffles the data around between all the processors, so that all data specific to a combination of airport/operation/year/month goes to one processor" and then produces a single file

In [ ]:
(
    df_fps_output
    .repartition("airport", "operation", "month")
    .write.option("header", True).partitionBy(["airport", "operation", "month"])
    .csv("CRAFT/" + year0 + "/flightplans", compression="gzip", mode="overwrite")
)

# Save LiteTrack Data

create the output litetrack dataframe by joining fps and lt dataframes

In [ ]:
df_lt_output = (
    df_fps_output.join(df_lt, on=["track_key"], how="inner")
    .withColumn("trackpoints", stringify_points_udf("points"))
    .select(
        "track_key",
        "start_date",
        "end_date",
        "orig",
        "dest",
        "mode_s_code",
        "callsign",
        "aircraft_type",
        "trackpoints",
        "airport",
        "month",
        "operation",
    )
)

Then, using the same partitioning scheme as before, save the trajectories pertaining to the FlightplanSeries data in the previous step.

In [ ]:
(
    df_lt_output
    .repartition("airport", "operation", "month")
    .write.option("header",True).partitionBy(["airport", "operation", "month"])
    .csv("CRAFT/" + year0 + "/litetracks", compression="gzip", mode="overwrite")
)